# nb03 - Performance: reading plans, AQE, skew, broadcast, caching

nb02 wrote a silver layer that is *deliberately* inefficient on disk: 60k orders
partitioned by `order_date`, each partition holding **6 tiny files** (one per source
part). This notebook is about seeing that cost in the plans, and letting Spark fix it.

**You will learn:** `explain("formatted")`, shuffle fundamentals,
`spark.sql.shuffle.partitions` vs **AQE coalescing**, skew detection & the AQE skew-join
split, broadcast vs sort-merge joins, and when `persist()` is actually worth it.

> Keep the **History Server** (http://localhost:18080) open - every job this notebook
> runs stays available for plan archaeology, and 4.2's UI can show you the initial vs
> final plan side-by-side for AQE queries.


In [1]:
import os
import time

from pyspark.sql import functions as F

from de_stack import get_spark, register_table

spark = get_spark("nb03-performance", shuffle_partitions=8)

for db, name in [("silver", "orders"), ("silver", "customers"), ("silver", "products")]:
    if not register_table(spark, db, name):
        raise RuntimeError(f"silver.{name} missing - run nb02_bronze_to_silver.ipynb first")

print("silver tables registered")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/13 11:35:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/10/13 11:35:33 WARN CreateNamespaceExec: Namespace silver was created concurrently. Ignoring.


26/10/13 11:35:33 WARN CreateNamespaceExec: Namespace silver was created concurrently. Ignoring.


silver tables registered


## 1. The small-files problem, literally

Each `order_date` partition should ideally be 1-3 files of 128 MB+. Ours are ~6 files
of a few KB - every one of them costs a task launch, a schema read, and a split.


In [2]:
partitions = sorted(d for d in os.listdir("/data/warehouse/silver.db/orders") if d.startswith("order_date="))
sample = f"/data/warehouse/silver.db/orders/{partitions[0]}"
print(partitions[0], "->", len([f for f in os.listdir(sample) if f.startswith('part-')]), "data files")


order_date=2026-09-13 -> 4 data files


## 2. Reading a plan

Three views of the same query. `formatted` is the one to learn: operators, their
statistics, and the partition counts at every exchange.


In [3]:
q = '''
    SELECT c.country,
           COUNT(*)    AS orders,
           ROUND(SUM(o.total_amount), 2) AS revenue
    FROM silver.orders o
    JOIN silver.customers c ON o.customer_id = c.customer_id
    GROUP BY c.country
    ORDER BY revenue DESC
    LIMIT 5
'''

df = spark.sql(q)
df.explain("formatted")
df.show(truncate=False)
print("driver UI (this query):", spark.sparkContext.uiWebUrl, "-> SQL tab")


26/10/13 11:35:35 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


== Physical Plan ==
AdaptiveSparkPlan (13)
+- TakeOrderedAndProject (12)
   +- HashAggregate (11)
      +- Exchange (10)
         +- HashAggregate (9)
            +- Project (8)
               +- BroadcastHashJoin Inner BuildRight (7)
                  :- Project (3)
                  :  +- Filter (2)
                  :     +- Scan parquet spark_catalog.silver.orders (1)
                  +- BroadcastExchange (6)
                     +- Filter (5)
                        +- Scan parquet spark_catalog.silver.customers (4)


(1) Scan parquet spark_catalog.silver.orders
Output [3]: [customer_id#6, total_amount#17, order_date#20]
Batched: true
Location: PreparedDeltaFileIndex [file:/data/warehouse/silver.db/orders]
PushedFilters: [IsNotNull(customer_id)]
ReadSchema: struct<customer_id:string,total_amount:double>

(2) Filter
Input [3]: [customer_id#6, total_amount#17, order_date#20]
Condition : isnotnull(customer_id#6)

(3) Project
Output [2]: [customer_id#6, total_amount#17]
Input [3]: [c

+-------+------+---------+
|country|orders|revenue  |
+-------+------+---------+
|IT     |6104  |452717.14|
|GB     |5978  |432860.29|
|BR     |5862  |432209.08|
|AU     |6026  |429922.25|
|FR     |6081  |429188.47|
+-------+------+---------+

driver UI (this query): http://jupyter:4040 -> SQL tab


## 3. Shuffles and `spark.sql.shuffle.partitions`

The rule of thumb is ~128 MB of input per partition, i.e. a 2 GB shuffle wants
~16 partitions. Our data is kilobytes, so the **200 default is absurd** - but AQE makes
over-provisioning harmless: `coalescePartitions` merges tiny output partitions at
runtime. Set 200 and watch the final partition count collapse.


In [4]:
spark.conf.set("spark.sql.shuffle.partitions", "200")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

agg = spark.sql("SELECT customer_id, COUNT(*) AS n, SUM(total_amount) AS amt FROM silver.orders GROUP BY customer_id")
agg.collect()  # materialize so AQE finishes its work

# ask the RDD how many partitions the FINAL plan produced
print("final partitions:", agg.rdd.getNumPartitions(), "(from spark.sql.shuffle.partitions=200)")


final partitions: 1 (from spark.sql.shuffle.partitions=200)


Now the same with AQE off - the answer is baked in at planning time:


In [5]:
spark.conf.set("spark.sql.adaptive.enabled", "false")

agg2 = spark.sql("SELECT customer_id, COUNT(*) AS n, SUM(total_amount) AS amt FROM silver.orders GROUP BY customer_id")
agg2.collect()
print("final partitions:", agg2.rdd.getNumPartitions(), "(AQE off: no coalescing)")

spark.conf.set("spark.sql.adaptive.enabled", "true")  # put it back


final partitions: 200 (AQE off: no coalescing)


## 4. Skew: one key to ruin a join

AQE's skew handling (`spark.sql.adaptive.skewJoin.enabled`, on by default) detects a
partition much larger than its siblings and splits it across tasks. We build a fact
table where one key owns ~10% of 2M rows, join it to a dimension, and compare the
executed plans with the feature disabled vs enabled.


In [6]:
from pyspark.sql.functions import when

fact = (spark.range(2_000_000)
        .select(when(F.col("id") % 1000 == 0, "HOT").otherwise(F.concat(F.lit("K"), F.col("id") % 99)).alias("key"),
                (F.col("id") % 100 + 1).alias("qty")))
dim = spark.createDataFrame([(f"K{i}", f"dim-{i:02d}") for i in range(99)] + [("HOT", "hot-dim")], "key STRING, label STRING")

fact.groupBy("key").count().orderBy(F.col("count").desc()).show(3)  # how hot is HOT?


+---+-----+
|key|count|
+---+-----+
|K18|20182|
|K69|20182|
|K96|20182|
+---+-----+
only showing top 3 rows


In [7]:
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")   # force a shuffle join

j_off = fact.join(dim, "key")
print("skew join OFF - look for the single long task in the SQL tab:")
j_off._jdf.queryExecution().executedPlan().toString().splitlines()[0]
count_off = j_off.count()


skew join OFF - look for the single long task in the SQL tab:


In [8]:
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

j_on = fact.join(dim, "key")
plan = j_on._jdf.queryExecution().executedPlan().toString()
count_on = j_on.count()

print("counts match:", count_off == count_on)
print("plan mentions AQE/skew adaptation:", "AdaptiveSparkPlan" in plan)
print("\nOpen the SQL tab for both jobs and compare the longest task duration;")
print("with skew handling the hot partition is sliced into several tasks.")


counts match: True
plan mentions AQE/skew adaptation: True

Open the SQL tab for both jobs and compare the longest task duration;
with skew handling the hot partition is sliced into several tasks.


## 5. Broadcast vs sort-merge join

`silver.customers` (10k rows, ~1 MB) is a textbook broadcast dimension. Watch the join
strategy flip in the plan when we lower the auto-broadcast threshold to nothing and then
hint it back.


In [9]:
join_q = '''
    SELECT /*+ BROADCAST(c) */ c.loyalty_tier, COUNT(*) AS n, ROUND(SUM(o.total_amount), 2) AS revenue
    FROM silver.orders o JOIN silver.customers c ON o.customer_id = c.customer_id
    GROUP BY c.loyalty_tier
'''

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
print("--- no broadcast allowed ---")
spark.sql(join_q).explain()

print("--- with BROADCAST hint ---")
hinted = spark.sql(join_q)
hinted.explain()
hinted.show(truncate=False)


--- no broadcast allowed ---


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[loyalty_tier#1140], functions=[count(1), sum(total_amount#1131)])
   +- Exchange hashpartitioning(loyalty_tier#1140, 200), ENSURE_REQUIREMENTS, [plan_id=1127]
      +- HashAggregate(keys=[loyalty_tier#1140], functions=[partial_count(1), partial_sum(total_amount#1131)])
         +- Project [total_amount#1131, loyalty_tier#1140]
            +- BroadcastHashJoin [customer_id#1120], [customer_id#1137], Inner, BuildRight, false, false
               :- Project [customer_id#1120, total_amount#1131]
               :  +- Filter isnotnull(customer_id#1120)
               :     +- FileScan parquet spark_catalog.silver.orders[customer_id#1120,total_amount#1131,order_date#1134] Batched: true, DataFilters: [isnotnull(customer_id#1120)], Format: Parquet, Location: PreparedDeltaFileIndex(1 paths)[file:/data/warehouse/silver.db/orders], PartitionFilters: [], PushedFilters: [IsNotNull(customer_id)], ReadSchema: struct<custom

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[loyalty_tier#1315], functions=[count(1), sum(total_amount#1306)])
   +- Exchange hashpartitioning(loyalty_tier#1315, 200), ENSURE_REQUIREMENTS, [plan_id=1193]
      +- HashAggregate(keys=[loyalty_tier#1315], functions=[partial_count(1), partial_sum(total_amount#1306)])
         +- Project [total_amount#1306, loyalty_tier#1315]
            +- BroadcastHashJoin [customer_id#1295], [customer_id#1312], Inner, BuildRight, false, false
               :- Project [customer_id#1295, total_amount#1306]
               :  +- Filter isnotnull(customer_id#1295)
               :     +- FileScan parquet spark_catalog.silver.orders[customer_id#1295,total_amount#1306,order_date#1309] Batched: true, DataFilters: [isnotnull(customer_id#1295)], Format: Parquet, Location: PreparedDeltaFileIndex(1 paths)[file:/data/warehouse/silver.db/orders], PartitionFilters: [], PushedFilters: [IsNotNull(customer_id)], ReadSchema: struct<custom

+------------+-----+----------+
|loyalty_tier|n    |revenue   |
+------------+-----+----------+
|bronze      |34921|2528046.44|
|silver      |18102|1300468.49|
|gold        |5774 |420040.31 |
+------------+-----+----------+



## 6. `persist()` - only for reuse

Caching costs memory and write time on first use; it pays off only when a DataFrame is
consumed more than once *in the same run*. The pattern: build the expensive frame once,
cache, then do several actions.


In [10]:
expensive = spark.sql('''
    SELECT o.order_id, o.order_date, o.total_amount, c.country, c.loyalty_tier
    FROM silver.orders o JOIN silver.customers c ON o.customer_id = c.customer_id
''')
expensive.persist()

t0 = time.perf_counter()
first = expensive.filter("total_amount > 100").count()
t1 = time.perf_counter()
second = expensive.groupBy("country").count().collect()
t2 = time.perf_counter()

print(f"action 1 (cold cache): {t1 - t0:.2f}s   rows>100: {first}")
print(f"action 2 (warm cache): {t2 - t1:.2f}s   countries: {len(second)}")

print("storage level:", expensive.storageLevel)
expensive.unpersist()


action 1 (cold cache): 6.38s   rows>100: 12220
action 2 (warm cache): 1.77s   countries: 10
storage level: Disk Memory Deserialized 1x Replicated


DataFrame[order_id: string, order_date: date, total_amount: double, country: string, loyalty_tier: string]

## 7. Your turn (open the UIs)

- **Master UI** (http://localhost:8080) - executors, cores, memory.
- **History Server** (http://localhost:18080) - every query above; 4.2's SQL page gives
  you a searchable, zoomable plan viewer and (for AQE queries) initial vs final plans
  side by side.
- Experiment: raise `GEN_EVENTS_PER_FILE` in `.env` and re-run nb05 to feel how
  `maxFilesPerTrigger` bounds batch size.

Next: **nb04_delta_lakehouse.ipynb** - giving these tables a time dimension with
Delta (MERGE, time travel, CDF, OPTIMIZE - which will also compact those 6-file partitions).
